# 01 — Chicago: Data Exploration

**Milestone 1** — pulling a live sample from the Chicago Data Portal to verify the real schema before we design the cross-city pipeline (Milestone 2).

- **Dataset:** ["Crimes - 2001 to Present"](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2), Chicago Data Portal (Socrata/SODA API)
- **Dataset ID:** `ijzp-q8t2`
- **Scope of this notebook:** a recent-window sample (not the full multi-million-row history — that full pull happens in a `scripts/` pipeline job in Milestone 2). Good enough here to confirm columns, types, null rates, and category values.

See `docs/DATA_SOURCES.md` for background on this dataset and `docs/ETHICS.md` for how this data may and may not be used.

In [1]:
import requests
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
DATASET_ENDPOINT = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
SAMPLE_SIZE = 20_000  # exploration sample, not the full historical dataset
PAGE_SIZE = 5_000     # Socrata's default max page size is 1000 unless raised; 5000 is safely within app-token-less limits for occasional use

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def fetch_socrata_sample(endpoint: str, total: int, page_size: int, order: str = ":id") -> pd.DataFrame:
    """Pull `total` rows from a Socrata endpoint via offset pagination, most-recent-first."""
    frames = []
    fetched = 0
    while fetched < total:
        limit = min(page_size, total - fetched)
        params = {
            "$limit": limit,
            "$offset": fetched,
            "$order": f"{order} DESC",
        }
        response = requests.get(endpoint, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()
        if not page:
            break
        frames.append(pd.DataFrame(page))
        fetched += len(page)
        if len(page) < limit:
            break  # ran out of rows before hitting `total`
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


chicago_df = fetch_socrata_sample(DATASET_ENDPOINT, SAMPLE_SIZE, PAGE_SIZE, order="date")
chicago_df.shape

(20000, 22)

In [4]:
chicago_df.dtypes

id                      object
case_number             object
date                    object
block                   object
iucr                    object
primary_type            object
description             object
location_description    object
arrest                    bool
domestic                  bool
beat                    object
district                object
ward                    object
community_area          object
fbi_code                object
x_coordinate            object
y_coordinate            object
year                    object
updated_on              object
latitude                object
longitude               object
location                object
dtype: object

In [5]:
chicago_df.head(3)

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14242025,JK309207,2026-06-25T00:00:00.000,008XX N KEDZIE AVE,0810,THEFT,OVER $500,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,1121,011,27,23,06,1154891,1905271,2026,2026-07-02T15:47:38.000,41.895877613,-87.706566627,"{'latitude': '41.895877613', 'longitude': '-87..."
1,14241153,JK308012,2026-06-25T00:00:00.000,074XX S MICHIGAN AVE,0820,THEFT,$500 AND UNDER,RESIDENCE,False,False,0323,003,6,69,06,1178450,1855617,2026,2026-07-02T15:47:38.000,41.759118274,-87.621551095,"{'latitude': '41.759118274', 'longitude': '-87..."
2,14241038,JK307916,2026-06-25T00:00:00.000,056XX S MICHIGAN AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,0225,002,20,40,14,1178119,1867657,2026,2026-07-02T15:47:38.000,41.792164799,-87.622399402,"{'latitude': '41.792164799', 'longitude': '-87..."


## Data quality checks

Everything below is checked live against the pulled sample — not assumed from documentation.

In [6]:
null_rates = (chicago_df.isna().mean() * 100).round(2).sort_values(ascending=False)
null_rates[null_rates > 0]

location_description    0.32
location                0.30
longitude               0.30
latitude                0.30
y_coordinate            0.30
x_coordinate            0.30
dtype: float64

In [7]:
chicago_df["date"] = pd.to_datetime(chicago_df["date"])
print("date range in this sample:", chicago_df["date"].min(), "to", chicago_df["date"].max())

date range in this sample: 2026-05-24 23:00:00 to 2026-06-25 00:00:00


In [8]:
chicago_df["primary_type"].value_counts().head(20)

primary_type
THEFT                               4277
BATTERY                             3930
CRIMINAL DAMAGE                     2173
ASSAULT                             1771
MOTOR VEHICLE THEFT                 1665
BURGLARY                            1346
OTHER OFFENSE                       1245
DECEPTIVE PRACTICE                   888
NARCOTICS                            500
CRIMINAL TRESPASS                    471
WEAPONS VIOLATION                    457
ROBBERY                              421
CRIMINAL SEXUAL ASSAULT              174
PUBLIC PEACE VIOLATION               145
OFFENSE INVOLVING CHILDREN           117
INTERFERENCE WITH PUBLIC OFFICER     100
SEX OFFENSE                           97
STALKING                              68
HOMICIDE                              40
ARSON                                 33
Name: count, dtype: int64

In [9]:
# `domestic` flags whether the incident was domestic-related -- directly relevant to our repeat-contact research question
chicago_df["domestic"].value_counts(dropna=False)

domestic
False    16062
True      3938
Name: count, dtype: int64

In [10]:
# community_area is a numeric code (1-77) -- we'll need Chicago's community area boundary file (data/external/)
# to turn this into a named neighborhood for maps and cross-city comparison in Milestone 2.
print("unique community_area codes in sample:", chicago_df["community_area"].nunique())
print("rows missing community_area:", chicago_df["community_area"].isna().sum())

unique community_area codes in sample: 77
rows missing community_area: 0


In [11]:
sample_path = RAW_DATA_DIR / "chicago_sample.csv"
chicago_df.to_csv(sample_path, index=False)
print(f"saved {len(chicago_df):,} rows to {sample_path}")

saved 20,000 rows to ../data/raw/chicago_sample.csv


## Findings

- **Sample:** 20,000 rows, ordered most-recent-first, spans 2026-05-24 to 2026-06-25 — roughly one month of citywide reports. At this reporting rate, Chicago alone generates on the order of several hundred thousand incidents per year, confirming we should pull the full pipeline via paginated/bulk export in Milestone 2 rather than loading it all into a notebook.

- **Nulls are minimal and concentrated in geography fields:** `location_description` (0.32%), and `location`/`latitude`/`longitude`/`x_coordinate`/`y_coordinate` (0.30% each, all missing together — same underlying rows lack geocoding). Everything else is fully populated. This is unusually clean for a government open dataset.

- **Top incident categories (`primary_type`)** in this sample: THEFT (4,277), BATTERY (3,930), CRIMINAL DAMAGE (2,173), ASSAULT (1,771), MOTOR VEHICLE THEFT (1,665), BURGLARY (1,346) dominate by volume. Lower-frequency but narratively important categories are present too: CRIMINAL SEXUAL ASSAULT (174), STALKING (68), HOMICIDE (40) — small numbers individually, but exactly the categories most relevant to the repeat-contact question, so category-level (not just top-N) breakdowns matter for the story.

- **`domestic` flag is genuinely useful:** 19.7% of this sample (3,938 / 20,000) is flagged domestic-related — a real, non-trivial share, and a native boolean field rather than something we'd have to infer. This is a strong candidate for a cross-city comparison, **but only Chicago has it** — NYC and SF's datasets don't expose an equivalent flag directly, so any cross-city "domestic-related" comparison will need to be inferred from offense-category text instead, which is weaker. Worth flagging explicitly to the reader as a data-availability limitation rather than pretending we have it everywhere.

- **Geography:** `community_area` is a numeric code, 77 unique values in this sample (matches Chicago's known 77 official community areas), zero missing. Clean field, but needs a join to Chicago's community area boundary/name file (`data/external/`) to become human-readable and mappable — that join happens in Milestone 2.

- **Nothing here changes `docs/ETHICS.md`** — no individual-level demographic fields in this dataset (confirmed: only geography, category, arrest/domestic flags, and IDs). The ethics policy on demographic data applies specifically to NYC, not Chicago.